# Task 0

## Part (a)
Loading pre-trained models

In [1]:
# Cell 1: Setup - Installations and Imports

# Install required third-party libraries
# !pip install pyJoules torchprofile tqdm pandas -q

# Jupyter magic command to automatically reload changed modules
%load_ext autoreload
%autoreload 2

# Standard and third-party library imports
import torch
import pandas as pd

# Local module imports from our project structure
from config import *
from utils.models import load_pretrained_model
from utils.data import get_dataloaders
from utils.profiler import get_model_size_mb, get_macs, evaluate_and_profile

print("Setup complete. All modules imported.")
print(f"Using device: {DEVICE}")

c:\Users\Fatim\anaconda3\envs\ml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete. All modules imported.
Using device: cuda


In [3]:
# Cell 2: Task 0(a) - Load Pretrained Models

print("--- Starting Task 0(a): Loading Pretrained Models ---")

# Load the VGG16-BN model for CIFAR-10
model_cifar10 = load_pretrained_model(MODEL_CIFAR10, DEVICE)
print("-" * 50)
# Load the VGG16-BN model for CIFAR-100
model_cifar100 = load_pretrained_model(MODEL_CIFAR100, DEVICE)

# Verify one of the models is loaded correctly by printing its architecture
if model_cifar10:
    print("\n--- CIFAR-10 VGG16-BN Architecture (Classifier only) ---")
    # print(model_cifar10.classifier)
    print(model_cifar10)

if model_cifar100:
    print("\n--- CIFAR-100 VGG16-BN Architecture (Classifier only) ---")
    # print(model_cifar100.classifier)
    print(model_cifar100)

--- Starting Task 0(a): Loading Pretrained Models ---
Loading pretrained model 'cifar10_vgg16_bn'...


Using cache found in C:\Users\Fatim/.cache\torch\hub\chenyaofo_pytorch-cifar-models_master


Model 'cifar10_vgg16_bn' loaded successfully and moved to cuda.
--------------------------------------------------
Loading pretrained model 'cifar100_vgg16_bn'...


Using cache found in C:\Users\Fatim/.cache\torch\hub\chenyaofo_pytorch-cifar-models_master


Model 'cifar100_vgg16_bn' loaded successfully and moved to cuda.

--- CIFAR-10 VGG16-BN Architecture (Classifier only) ---
VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(i

## Part (b) and (c)
Loading CIFAR dataset and building data loaders

In [3]:
# Cell 3: Task 0(b) & 0(c) - Load Datasets and Build Dataloaders

print("\n--- Starting Task 0(b) & 0(c): Loading Datasets and Dataloaders ---")

# Create dataloaders for CIFAR-10
train_loader_cifar10, test_loader_cifar10 = get_dataloaders(
    "cifar10", DATA_DIR, BATCH_SIZE, NUM_WORKERS, CIFAR10_STATS
)
print("-" * 50)
# Create dataloaders for CIFAR-100
train_loader_cifar100, test_loader_cifar100 = get_dataloaders(
    "cifar100", DATA_DIR, BATCH_SIZE, NUM_WORKERS, CIFAR100_STATS
)

# Verification step: Inspect a single batch to ensure correctness
print("\n--- Verifying a batch from CIFAR-10 test loader ---")
images, labels = next(iter(test_loader_cifar10))
print(f"Image batch shape: {images.shape}")
print(f"Labels batch shape: {labels.shape}")


--- Starting Task 0(b) & 0(c): Loading Datasets and Dataloaders ---
Loading 'CIFAR10' dataset...
Files already downloaded and verified
Files already downloaded and verified
Train dataset size: 50000
Test dataset size: 10000
--------------------------------------------------
Loading 'CIFAR100' dataset...
Files already downloaded and verified
Files already downloaded and verified
Train dataset size: 50000
Test dataset size: 10000

--- Verifying a batch from CIFAR-10 test loader ---
Image batch shape: torch.Size([256, 3, 32, 32])
Labels batch shape: torch.Size([256])


## Task (d) and (e)
Profiling and verifying baseline accuracy

In [4]:
# Cell 4: Task 0(d) & 0(e) - Profile Models and Verify Baseline Accuracy

print("\n--- Starting Task 0(d) & 0(e): Profiling and Accuracy Verification ---")

results_data = []
all_models_info = [
    {
        "dataset": "CIFAR-10",
        "model": model_cifar10,
        "train_loader": train_loader_cifar10,
        "test_loader": test_loader_cifar10,
    },
    {
        "dataset": "CIFAR-100",
        "model": model_cifar100,
        "train_loader": train_loader_cifar100,
        "test_loader": test_loader_cifar100,
    },
]

for item in all_models_info:
    model, dataset = item["model"], item["dataset"]
    print(f"\n--- Profiling VGG16-BN on {dataset} ---")

    # 1. Model Size (Static)
    model_size = get_model_size_mb(model)

    # 2. MACs (Static)
    macs = get_macs(model, (3, 32, 32), DEVICE)

    # 3. Dynamic Profiling (Latency, Memory, Energy) and Accuracy on Test Set
    test_metrics = evaluate_and_profile(
        model=model,
        loader=item["test_loader"],
        device=DEVICE,
        num_profiling_batches=NUM_PROFILING_BATCHES,
        description=f"Profiling {dataset} Test Set",
    )

    # 4. Accuracy on Train Set (for baseline verification)
    train_metrics = evaluate_and_profile(
        model=model,
        loader=item["train_loader"],
        device=DEVICE,
        num_profiling_batches=0,  # No need to re-profile, just get accuracy
        description=f"Evaluating {dataset} Train Set",
    )

    results_data.append(
        {
            "Model": "VGG16-BN",
            "Dataset": dataset,
            "Model Size (MB)": model_size,
            "MACs (G)": macs / 1e9,
            "Peak Memory (MB)": test_metrics["peak_memory_mb"],
            "Latency (ms/batch)": test_metrics["avg_latency_ms"],
            "Energy (mJ/batch)": test_metrics["avg_energy_mj"],
            "Test Top-1 Acc (%)": test_metrics["top1_acc"],
            "Test Top-5 Acc (%)": test_metrics["top5_acc"],
            "Train Top-1 Acc (%)": train_metrics["top1_acc"],
        }
    )

print("\nProfiling and evaluation complete.")


--- Starting Task 0(d) & 0(e): Profiling and Accuracy Verification ---

--- Profiling VGG16-BN on CIFAR-10 ---


Evaluating CIFAR-10 Train Set: 100%|██████████| 196/196 [01:27<00:00,  2.23it/s, Top-1 Acc=100.00%]



--- Profiling VGG16-BN on CIFAR-100 ---


Evaluating CIFAR-100 Train Set: 100%|██████████| 196/196 [01:03<00:00,  3.08it/s, Top-1 Acc=99.93%] 



Profiling and evaluation complete.


## Results

In [5]:
# Cell 5: Deliverables - Present and Save Results

print("\n--- Baseline Profiling and Accuracy Results ---")

# Create a pandas DataFrame from the collected results
results_df = pd.DataFrame(results_data)

# Set display options for better readability in the notebook
pd.options.display.float_format = "{:.2f}".format

# Print the final table to the console
print(results_df.to_string())

# Save the results to a markdown file for the official submission
report_filename = "task0_baseline_results.md"
with open(report_filename, "w") as f:
    f.write("# Task 0: Baseline Profiling Results\n\n")
    f.write(results_df.to_markdown(index=False))

print(f"\nResults have been successfully saved to '{report_filename}'")


--- Baseline Profiling and Accuracy Results ---
      Model    Dataset  Model Size (MB)  MACs (G)  Peak Memory (MB)  Latency (ms/batch)  Energy (mJ/batch)  Test Top-1 Acc (%)  Test Top-5 Acc (%)  Train Top-1 Acc (%)
0  VGG16-BN   CIFAR-10            58.25      0.31            321.68               39.16                  0               94.15               99.71               100.00
1  VGG16-BN  CIFAR-100            58.43      0.31            321.77               33.37                  0               74.00               90.54                99.93

Results have been successfully saved to 'task0_baseline_results.md'
